Build Simulator

In [60]:
%run station_classification.ipynb
simulate = Simulator(sim_envelops, last_updates, Metropolis_metric)


selected times per day for simulation [0, 15, 33, 48, 63, -1]
enevelops input shape (stations, days, times per day) (10, 7, 73)
selected shape (stations, days, times per day) (10, 7, 6)
simulating null policy: [0 0 0 0 0 0 0]
simulating  massive extraction policy: [-15 -15 -15 -15 -15 -15 -15]
priority stations==============================
['P-Full' 'P-Full' 'P-Full' 'P-Full' 'P-Full' 'P-Full' 'P-Full' '------'
 '------' 'P-Full']
buffer stations and corresponding day==========
[['B-full' 'B-full' 'B-full' 'B-full' '------' 'B-full' '------']
 ['------' 'B-full' '------' '------' '------' '------' 'B-full']
 ['------' '------' '------' '------' '------' '------' '------']
 ['------' '------' '------' '------' 'B-full' 'B-full' '------']
 ['------' '------' '------' '------' '------' '------' 'B-full']
 ['------' 'B-full' 'B-full' '------' '------' '------' '------']
 ['B-full' 'B-full' 'B-full' 'B-full' 'B-full' 'B-full' 'B-full']
 ['B-empt' '------' 'B-empt' '------' '------' 'B-empt

Build all possible strategyes, encoded as bits

In [6]:
def int_to_binary_matrix(indices, n_bits):
    return (indices[:, None] & (1 << np.arange(n_bits)[::-1]) > 0).astype(int)

# 3. Simulation
n_futur_days = 7
strat_idx = np.arange(2**n_futur_days)
strat_bits = int_to_binary_matrix(strat_idx, n_futur_days)
print(strat_bits[:10])

[[0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1]
 [0 0 0 0 0 1 0]
 [0 0 0 0 0 1 1]
 [0 0 0 0 1 0 0]
 [0 0 0 0 1 0 1]
 [0 0 0 0 1 1 0]
 [0 0 0 0 1 1 1]
 [0 0 0 1 0 0 0]
 [0 0 0 1 0 0 1]]


We introduce natural notion of Pareto domination

In [19]:
import numpy as np
from typing import Dict, Set, Tuple

def parse_bits(s_sequence) -> Tuple[int, ...]:
    """Converts a sequence of bits (e.g., numpy array, list) to a tuple of integers."""
    # Ensure all elements are integers and it's an immutable tuple
    return tuple(map(int, s_sequence))

def build_partial_orders(all_strats_matrix):
    """
    Builds partial order relationships (dominance) between a set of strategies.
    `all_strats_matrix` is expected to be a 2D numpy array where each row is a strategy.
    """
    # Convert each strategy (row) from numpy array to a hashable tuple
    bits_map: Dict[Tuple[int, ...], Tuple[int, ...]] = {parse_bits(row): parse_bits(row) for row in all_strats_matrix}

    INF: Dict[Tuple[int, ...], Set[Tuple[int, ...]]] = {s: set() for s in bits_map.keys()} # Strategies dominated by s
    SUP: Dict[Tuple[int, ...], Set[Tuple[int, ...]]] = {s: set() for s in bits_map.keys()} # Strategies that dominate s

    for s_tuple in bits_map.keys():
        b_tuple = bits_map[s_tuple] # Get the bit values for s_tuple
        for t_tuple in bits_map.keys():
            if t_tuple == s_tuple:
                continue
            bt_tuple = bits_map[t_tuple] # Get the bit values for t_tuple

            # Check if t_tuple is "less than or equal to" s_tuple everywhere (s_tuple dominates t_tuple)
            if all(x <= y for x, y in zip(bt_tuple, b_tuple)):
                INF[s_tuple].add(t_tuple) # s_tuple dominates t_tuple
            # Check if t_tuple is "greater than or equal to" s_tuple everywhere (t_tuple dominates s_tuple)
            if all(x >= y for x, y in zip(bt_tuple, b_tuple)):
                SUP[s_tuple].add(t_tuple) # t_tuple dominates s_tuple

    return INF, SUP

# Example usage:
print("exemple strategy (numpy array):", strat_bits[30])

# First, build the partial orders for ALL strategies
all_inf_orders, all_sup_orders = build_partial_orders(strat_bits)

example_strategy_tuple = parse_bits(strat_bits[30])
print(
    "Strategies dominated by exemple strategy:",
    list(all_inf_orders.get(example_strategy_tuple, set()))[:5]
)
print(
    "Strategies that dominate example strategy:",
    list(all_sup_orders.get(example_strategy_tuple, set()))[:5]
)

exemple strategy (numpy array): [0 0 1 1 1 1 0]
Strategies dominated by exemple strategy: [(0, 0, 1, 0, 1, 1, 0), (0, 0, 1, 1, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0), (0, 0, 0, 1, 1, 1, 0), (0, 0, 1, 0, 1, 0, 0)]
Strategies that dominate example strategy: [(1, 1, 1, 1, 1, 1, 1), (0, 1, 1, 1, 1, 1, 0), (1, 1, 1, 1, 1, 1, 0), (1, 0, 1, 1, 1, 1, 1), (1, 0, 1, 1, 1, 1, 0)]


### Strategy simulation

Here we simulate all **2⁷ = 128** possible strategies one at a time
(using a toy model with only **10 stations**).

For larger datasets, it would be preferable to build a simulator able to
process **batches of strategies**, allowing **parallel computation**.

In [76]:
Noperation_truck = {"injection": 15, "extraction": -15}

Exemple call of simulator

In [97]:
help(simulate.__call__)
policy = np.full(7, -15)
print("simulating  massive extraction policy:", policy)
print("output results", simulate(policy).keys())
print("exemple: test_metric", simulate(policy)["test_metric"])

Help on method __call__ in module __main__:

__call__(policy, remove_midnight={'current': True, 'next': True}, output='all') method of __main__.Simulator instance
    Call self as a function.

simulating  massive extraction policy: [-15 -15 -15 -15 -15 -15 -15]
output results dict_keys(['-', 'shape', 'stock', 'feasible', 'test_metric', 'test_buffer'])
exemple: test_metric {'empty': array([False, False, False, False, False, False, False, False, False,
       False]), 'full': array([ True,  True,  True,  True, False,  True, False, False, False,
       False])}


Exemple: Evaluate extraction policys (here each operation removes Noperation = 15 bikes)

In [98]:
policy_ok = np.zeros(
    (len(strat_bits), n_stations), dtype = bool
    )  #shape (n_strategys, n_stations)


for strat_idx, strat in enumerate(strat_bits):
  policy = strat*Noperation_truck["injection"] #shape n_days
  sim = simulate(policy)

  policy_ok[strat_idx] = (
      (~sim["test_metric"]["empty"]) &
      (~sim["test_metric"]["full"]) &
      (sim["feasible"])
  ) #shape (n_stations)

strat_idx, station_idx = np.where(policy_ok)

print("all admissibile (strategie, station):")
print("==========================\n")
for s, st in zip(strat_idx, station_idx):
    print(f"strategy {s} = {strat_bits[s]}  ->  station {st}")

all admissibile (strategie, station):

strategy 0 = [0 0 0 0 0 0 0]  ->  station 7
strategy 0 = [0 0 0 0 0 0 0]  ->  station 8
strategy 1 = [0 0 0 0 0 0 1]  ->  station 8
strategy 2 = [0 0 0 0 0 1 0]  ->  station 7
strategy 16 = [0 0 1 0 0 0 0]  ->  station 7
strategy 16 = [0 0 1 0 0 0 0]  ->  station 8
strategy 17 = [0 0 1 0 0 0 1]  ->  station 8
strategy 18 = [0 0 1 0 0 1 0]  ->  station 7
strategy 32 = [0 1 0 0 0 0 0]  ->  station 8
strategy 33 = [0 1 0 0 0 0 1]  ->  station 8
strategy 48 = [0 1 1 0 0 0 0]  ->  station 8
strategy 49 = [0 1 1 0 0 0 1]  ->  station 8
strategy 64 = [1 0 0 0 0 0 0]  ->  station 7
strategy 64 = [1 0 0 0 0 0 0]  ->  station 8
strategy 65 = [1 0 0 0 0 0 1]  ->  station 8
strategy 66 = [1 0 0 0 0 1 0]  ->  station 7
strategy 80 = [1 0 1 0 0 0 0]  ->  station 7
strategy 80 = [1 0 1 0 0 0 0]  ->  station 8
strategy 81 = [1 0 1 0 0 0 1]  ->  station 8
strategy 82 = [1 0 1 0 0 1 0]  ->  station 7
strategy 96 = [1 1 0 0 0 0 0]  ->  station 8
strategy 97 = [1 1 0

PARETO-Frontiers = Compact representation of admissibility

In [104]:
print("\nCriterion:")
print("down_frontier <= policy <= up_frontier\n")


Criterion:
down_frontier <= policy <= up_frontier



In [105]:
# Precompute tuple version of strategies
strat_tuples = [parse_bits(s) for s in strat_bits]

for station_idx in range(n_stations):

    admissible_idx = np.where(policy_ok[:, station_idx])[0]
    admissible_set = {strat_tuples[i] for i in admissible_idx}

    down_frontier = []
    up_frontier = []

    for s_idx in admissible_idx:
        s_tuple = strat_tuples[s_idx]

        # dominated strategies that are admissible
        dominated = all_inf_orders[s_tuple] & admissible_set

        # dominating strategies that are admissible
        dominating = all_sup_orders[s_tuple] & admissible_set

        # remove itself if present
        dominated = {x for x in dominated if x != s_tuple}
        dominating = {x for x in dominating if x != s_tuple}

        if len(dominated) == 0:
            down_frontier.append(s_idx)

        if len(dominating) == 0:
            up_frontier.append(s_idx)

    print(f"\nStation {station_idx}")
    print("=" * 40)

    print("Down frontier:")
    for s in down_frontier:
        print(f"  strategy {s} = {strat_bits[s]}")

    print("\nUp frontier:")
    for s in up_frontier:
        print(f"  strategy {s} = {strat_bits[s]}")

    print("\nAdmissible policies:")
    for s in admissible_idx[:5]:
        print(f"  strategy {s} = {strat_bits[s]}")
    print("\n...")



Station 0
Down frontier:

Up frontier:

Admissible policies:

...

Station 1
Down frontier:

Up frontier:

Admissible policies:

...

Station 2
Down frontier:

Up frontier:

Admissible policies:

...

Station 3
Down frontier:

Up frontier:

Admissible policies:

...

Station 4
Down frontier:

Up frontier:

Admissible policies:

...

Station 5
Down frontier:

Up frontier:

Admissible policies:

...

Station 6
Down frontier:

Up frontier:

Admissible policies:

...

Station 7
Down frontier:
  strategy 0 = [0 0 0 0 0 0 0]

Up frontier:
  strategy 82 = [1 0 1 0 0 1 0]

Admissible policies:
  strategy 0 = [0 0 0 0 0 0 0]
  strategy 2 = [0 0 0 0 0 1 0]
  strategy 16 = [0 0 1 0 0 0 0]
  strategy 18 = [0 0 1 0 0 1 0]
  strategy 64 = [1 0 0 0 0 0 0]

...

Station 8
Down frontier:
  strategy 0 = [0 0 0 0 0 0 0]

Up frontier:
  strategy 113 = [1 1 1 0 0 0 1]

Admissible policies:
  strategy 0 = [0 0 0 0 0 0 0]
  strategy 1 = [0 0 0 0 0 0 1]
  strategy 16 = [0 0 1 0 0 0 0]
  strategy 17 = [0 0 1 